# Volume-Conditioned Short Momentum vs EQW

1M短期モメンタムが出来高条件で継続/反転する効果を、`EQW` 基準で検証する。


## 1) 研究目的・仮説・数式

### 目的
- 1M短期モメンタムを、ETF出来高の状態に応じて連続的に反転させる。
- 比較対象は `EQW` のみ（既存戦略との多重比較は行わない）。

### 仮説
- 出来高が高い局面では、短期成分は順張り優位から逆張り優位へ遷移しやすい。

### モデル
出来高標準化（ETF別）:

$$
z_{i,t}=rac{\log(1+V_{i,t})-\mu_{i,t}^{(W)}}{\sigma_{i,t}^{(W)}}
$$

混合確率（シグモイド）:

$$
\pi_{i,t}=\sigma\left(a(z_{i,t}-c)ight),\quad \sigma(x)=rac{1}{1+e^{-x}}
$$

短期成分の反転係数:

$$
\kappa_{i,t}=1-2\pi_{i,t}\in[-1,1]
$$

予測合成（1M成分のみ反転対象）:

$$
\hat r^{mix}_{i,t}=\kappa_{i,t}\hat r^{short}_{i,t}+\hat r^{long}_{i,t}
$$

診断回帰:

$$
r_{i,t+1}=eta_0+eta_1\hat r^{short}_{i,t}+eta_2 z_{i,t}+eta_3(\hat r^{short}_{i,t}z_{i,t})+arepsilon_{i,t+1}
$$

期待符号: $eta_3<0$（出来高上昇時に短期順張り効果が弱まり反転寄与が増える）。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

np.random.seed(42)

CONFIG = {
    # Data
    'start_date': '2010-07-23',
    'end_date': None,
    'tickers': [
        'AGNG','AIQ','AQWA','BITS','BKCH','BOTZ','BUG','CHPX','CLOU','CTEC','DRIV','DTCR',
        'EBIZ','FINX','GNOM','HEAL','HERO','HYDR','IPAV','KROP','LIT','MILN','PAVE','RNRG',
        'SHLD','SNSR','SOCL','ZAP'
    ],
    'local_csv_path': 'data/theme_etf.csv',
    'rebalance': 'W-FRI',

    # Backtest
    'cost_bps': 10,

    # PCA
    'pca_window_months': 60,
    'n_components': 8,
    'min_obs_per_window': 24,

    # Momentum
    'factor_mom_lookbacks': [1, 12],
    'factor_mom_skip_months': 1,
    'combine_weights': {1: 0.5, 12: 0.5},
    'top_m_factors': 3,

    # Volume-conditioned mix (fixed setting)
    'volmix_a': 1.2,
    'volmix_c': 0.5,
    'volmix_volume_z_window_months': 6,

    # Explicitly isolate from heat framework
    'run_theme_heat_variants': False,
}

def load_data(config):
    tickers = config['tickers']
    start = pd.to_datetime(config.get('start_date')) if config.get('start_date') else None
    end = pd.to_datetime(config.get('end_date')) if config.get('end_date') else None

    raw = pd.read_csv(config['local_csv_path'], header=[0, 1], index_col=0, parse_dates=True)
    px_d = raw['Adj Close'].reindex(columns=tickers).sort_index()
    if 'Volume' in raw.columns.get_level_values(0):
        vol_d = raw['Volume'].reindex(columns=tickers).sort_index()
    else:
        vol_d = pd.DataFrame(index=px_d.index, columns=px_d.columns)

    if start is not None:
        px_d = px_d.loc[px_d.index >= start]
        vol_d = vol_d.loc[vol_d.index >= start]
    if end is not None:
        px_d = px_d.loc[px_d.index <= end]
        vol_d = vol_d.loc[vol_d.index <= end]

    return px_d, vol_d


px_d, vol_d = load_data(CONFIG)
print('data loaded:', px_d.shape, vol_d.shape)


In [ ]:
def _convert_months_to_periods(month_value, scale):
    return max(1, int(round(float(month_value) * scale)))


def resolve_runtime_params(config):
    rebalance = str(config.get('rebalance', 'M')).upper()
    if rebalance == 'M':
        rule = 'ME'
        periods_per_year = 12
        scale = 1.0
        label = 'Monthly (EOM)'
    elif rebalance.startswith('W-'):
        rule = rebalance
        periods_per_year = 52
        scale = 52.0 / 12.0
        label = f'Weekly ({rebalance})'
    else:
        raise ValueError('rebalance must be M or W-XXX')

    pca_window = _convert_months_to_periods(config['pca_window_months'], scale)
    min_obs = _convert_months_to_periods(config['min_obs_per_window'], scale)
    skip_periods = _convert_months_to_periods(config['factor_mom_skip_months'], scale)

    lookbacks_months = [int(x) for x in config['factor_mom_lookbacks']]
    lookbacks_periods = [_convert_months_to_periods(x, scale) for x in lookbacks_months]

    raw_cw = {int(k): float(v) for k, v in config['combine_weights'].items()}
    combined = {}
    for lb_m, lb_p in zip(lookbacks_months, lookbacks_periods):
        combined[lb_p] = combined.get(lb_p, 0.0) + raw_cw.get(lb_m, 0.0)

    if sum(combined.values()) <= 0:
        unique_lb = sorted(set(lookbacks_periods))
        combined = {k: 1.0 / len(unique_lb) for k in unique_lb}
    else:
        s = sum(combined.values())
        combined = {k: v / s for k, v in combined.items()}

    return {
        'rule': rule,
        'periods_per_year': periods_per_year,
        'scale': scale,
        'label': label,
        'pca_window': pca_window,
        'min_obs': min_obs,
        'skip_periods': skip_periods,
        'lookbacks_periods': sorted(set(lookbacks_periods)),
        'combine_weights_periods': combined,
    }


def resolve_volume_mix_runtime_params(config, runtime):
    return {
        'volume_z_window_periods': _convert_months_to_periods(config.get('volmix_volume_z_window_months', 6), runtime['scale'])
    }


def to_periodic(px_d, vol_d, rule):
    px_p = px_d.resample(rule).last()
    vol_p = vol_d.resample(rule).sum(min_count=1)
    ret_p = px_p.pct_change()
    return px_p, ret_p, vol_p


RUNTIME = resolve_runtime_params(CONFIG)
RUNTIME_VM = resolve_volume_mix_runtime_params(CONFIG, RUNTIME)
px_p, ret_p, vol_p = to_periodic(px_d, vol_d, RUNTIME['rule'])

print(RUNTIME['label'], '| obs=', len(ret_p), '| volume_z_window_periods=', RUNTIME_VM['volume_z_window_periods'])


In [ ]:
def build_rolling_pca(ret_p, runtime, n_components=8):
    window = runtime['pca_window']
    min_obs = runtime['min_obs']

    factor_cols = [f'F{i+1}' for i in range(n_components)]
    factor_ret = pd.DataFrame(index=ret_p.index, columns=factor_cols, dtype=float)

    loadings_by_t = {}
    eligibility_mask_by_t = {}

    prev_load = None

    for i, t in enumerate(ret_p.index):
        if i < window - 1:
            continue

        win = ret_p.iloc[i - window + 1 : i + 1]
        obs = win.notna().sum()
        elig = obs >= min_obs
        elig = elig & ret_p.loc[t].notna()
        elig = elig.reindex(ret_p.columns).fillna(False)
        eligibility_mask_by_t[t] = elig

        if int(elig.sum()) < 2:
            continue

        x = win.loc[:, elig.index[elig.values]].copy()
        x = x.apply(lambda col: col.fillna(col.mean()), axis=0).fillna(0.0)

        mu = x.mean(axis=0)
        sd = x.std(axis=0, ddof=0).replace(0.0, 1.0)
        x_std = (x - mu) / sd

        k = min(n_components, x_std.shape[1], x_std.shape[0])
        if k < 1:
            continue

        pca = PCA(n_components=k, random_state=42)
        scores = pca.fit_transform(x_std.values)
        loadings = pd.DataFrame(pca.components_.T, index=x_std.columns, columns=[f'F{i+1}' for i in range(k)])

        if prev_load is not None:
            common_assets = loadings.index.intersection(prev_load.index)
            common_factors = loadings.columns.intersection(prev_load.columns)
            for fc in common_factors:
                if len(common_assets) == 0:
                    continue
                anchor = float((loadings.loc[common_assets, fc] * prev_load.loc[common_assets, fc]).sum())
                if anchor < 0:
                    loadings[fc] *= -1.0
                    col_idx = list(loadings.columns).index(fc)
                    scores[:, col_idx] *= -1.0

        prev_load = loadings.copy()

        f_t = pd.Series(scores[-1, :], index=loadings.columns)
        factor_ret.loc[t, f_t.index] = f_t.values
        loadings_by_t[t] = loadings

    return factor_ret, loadings_by_t, eligibility_mask_by_t


factor_ret, loadings_by_t, eligibility_mask_by_t = build_rolling_pca(
    ret_p=ret_p,
    runtime=RUNTIME,
    n_components=CONFIG['n_components'],
)

print('factor_ret shape:', factor_ret.shape, '| loadings dates:', len(loadings_by_t))


In [ ]:
def compute_factor_mom_component(factor_ret, lookback_period, skip_periods):
    L = int(lookback_period)
    start_lag = 1
    if skip_periods > 0 and L > (skip_periods + 1):
        start_lag = int(skip_periods) + 1

    lagged = [factor_ret.shift(j) for j in range(start_lag, L + 1)]
    if not lagged:
        return factor_ret * np.nan
    return sum(lagged)


def _normalize_abs_signal(sig):
    s = sig.copy().fillna(0.0)
    denom = s.abs().sum()
    return s / denom if denom > 0 else s * 0.0


def build_factor_signal_components(mom_short_t, mom_long_t, top_m, select_by_abs=True):
    combined = (mom_short_t.fillna(0.0) + mom_long_t.fillna(0.0)).dropna()
    out_short = pd.Series(0.0, index=mom_short_t.index, dtype=float)
    out_long = pd.Series(0.0, index=mom_long_t.index, dtype=float)

    if len(combined) == 0:
        return out_short, out_long

    k = min(int(top_m), len(combined))
    if k <= 0:
        return out_short, out_long

    if select_by_abs:
        picked = combined.abs().nlargest(k).index
    else:
        picked = combined.nlargest(k).index

    out_short.loc[picked] = mom_short_t.reindex(picked).fillna(0.0).values
    out_long.loc[picked] = mom_long_t.reindex(picked).fillna(0.0).values

    return _normalize_abs_signal(out_short), _normalize_abs_signal(out_long)


def map_signal_to_etf_prediction(exposure_t, signal_t, elig_t):
    all_assets = elig_t.index
    if exposure_t is None or exposure_t.empty:
        return pd.Series(0.0, index=all_assets, dtype=float)

    fac = exposure_t.columns.intersection(signal_t.index)
    if len(fac) == 0:
        return pd.Series(0.0, index=all_assets, dtype=float)

    r_raw = exposure_t[fac].dot(signal_t.reindex(fac).fillna(0.0))
    out = pd.Series(0.0, index=all_assets, dtype=float)
    out.loc[r_raw.index] = r_raw.values
    out.loc[~elig_t] = 0.0
    return out.fillna(0.0)


lookbacks = sorted(RUNTIME['lookbacks_periods'])
short_lb = lookbacks[0]
long_lbs = [x for x in lookbacks if x != short_lb]

mom_short = compute_factor_mom_component(factor_ret, short_lb, RUNTIME['skip_periods'])

if len(long_lbs) == 0:
    mom_long = pd.DataFrame(0.0, index=factor_ret.index, columns=factor_ret.columns)
else:
    cw = RUNTIME['combine_weights_periods']
    wsum = sum(cw.get(lb, 0.0) for lb in long_lbs)
    if wsum <= 0:
        lw = {lb: 1.0 / len(long_lbs) for lb in long_lbs}
    else:
        lw = {lb: cw.get(lb, 0.0) / wsum for lb in long_lbs}

    mom_long = sum(
        lw[lb] * compute_factor_mom_component(factor_ret, lb, RUNTIME['skip_periods'])
        for lb in long_lbs
    )

print('short_lb(period):', short_lb, '| long_lbs(period):', long_lbs)


In [ ]:
def compute_volume_zscore_etf(vol_p, t, window_periods, assets):
    assets = list(assets)
    if len(assets) == 0:
        return pd.Series(dtype=float)
    if t not in vol_p.index:
        return pd.Series(0.0, index=assets, dtype=float)

    pos = vol_p.index.get_loc(t)
    if isinstance(pos, slice):
        pos = pos.stop - 1
    elif isinstance(pos, np.ndarray):
        pos = int(np.where(pos)[0][-1])

    start = max(0, int(pos) - int(window_periods) + 1)
    win = np.log1p(vol_p.iloc[start : int(pos) + 1].reindex(columns=assets).astype(float))
    win = win.replace([np.inf, -np.inf], np.nan)

    if len(win) == 0:
        return pd.Series(0.0, index=assets, dtype=float)

    cur = win.iloc[-1]
    mu = win.mean(axis=0)
    sd = win.std(axis=0, ddof=0).replace(0.0, np.nan)

    z = (cur - mu) / sd
    return z.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)


def compute_pi_from_volume_z(z_s, a=1.2, c=0.5):
    x = float(a) * (z_s.fillna(0.0).astype(float) - float(c))
    return 1.0 / (1.0 + np.exp(-x))


def combine_short_long_prediction(rhat_short, rhat_long, pi_s):
    pi = pi_s.reindex(rhat_short.index).fillna(0.5).clip(lower=0.0, upper=1.0)
    kappa = 1.0 - 2.0 * pi
    mix = kappa * rhat_short + rhat_long.reindex(rhat_short.index).fillna(0.0)
    return mix, kappa


def _allocate_proportional_long_only(score, elig_t):
    s = score.reindex(elig_t.index).fillna(0.0)
    s.loc[~elig_t] = 0.0
    pos = s[s > 0]
    out = pd.Series(0.0, index=elig_t.index, dtype=float)
    if len(pos) > 0:
        out.loc[pos.index] = pos.values
        out = out / out.sum()
    return out


def compute_turnover(w_t, w_prev):
    return 0.5 * np.abs(w_t - w_prev).sum()


def next_period_index(idx, t):
    if t not in idx:
        return None
    pos = idx.get_loc(t)
    if isinstance(pos, slice):
        pos = pos.stop - 1
    elif isinstance(pos, np.ndarray):
        pos = int(np.where(pos)[0][-1])
    nxt = int(pos) + 1
    if nxt >= len(idx):
        return None
    return idx[nxt]


def run_backtest(weights_by_t, ret_p, cost_bps):
    if weights_by_t.empty:
        idx = pd.DatetimeIndex([], name='period')
        empty = pd.Series(index=idx, dtype=float)
        return empty.copy(), empty.copy(), empty.copy(), empty.copy(), empty.copy()

    weights_by_t = weights_by_t.sort_index().fillna(0.0)
    tickers = list(weights_by_t.columns)

    decision_periods = []
    applied_periods = []
    gross_vals = []
    net_vals = []
    turn_vals = []

    w_prev = pd.Series(0.0, index=tickers)

    for t, w_t in weights_by_t.iterrows():
        apply_t = next_period_index(ret_p.index, t)
        if apply_t is None:
            continue

        r_next = ret_p.loc[apply_t].reindex(tickers).fillna(0.0)
        w_t = w_t.reindex(tickers).fillna(0.0)

        turnover = compute_turnover(w_t, w_prev)
        gross = float(np.dot(w_t.values, r_next.values))
        net = gross - turnover * (cost_bps / 10000.0)

        decision_periods.append(t)
        applied_periods.append(apply_t)
        gross_vals.append(gross)
        net_vals.append(net)
        turn_vals.append(turnover)

        w_prev = w_t

    idx = pd.DatetimeIndex(applied_periods, name='period')
    gross_s = pd.Series(gross_vals, index=idx, name='gross')
    net_s = pd.Series(net_vals, index=idx, name='net')
    turn_s = pd.Series(turn_vals, index=idx, name='turnover')

    equity = (1.0 + net_s).cumprod().rename('equity')
    peak = equity.cummax()
    dd = (equity / peak - 1.0).rename('drawdown')

    decision_idx = pd.DatetimeIndex(decision_periods, name='decision_period')
    for s in (gross_s, net_s, turn_s, equity, dd):
        s.attrs['decision_periods'] = decision_idx

    return gross_s, net_s, turn_s, equity, dd


def run_volume_conditioned_strategy(
    ret_p, vol_p, loadings_by_t, eligibility_mask_by_t,
    mom_short, mom_long, top_m, volume_z_window_periods, a, c, cost_bps
):
    all_tickers = ret_p.columns.tolist()

    weights_dict = {}
    diag = {
        'rhat_short': {},
        'rhat_long': {},
        'rhat_mix': {},
        'z': {},
        'pi': {},
        'kappa': {},
        'elig': {},
    }

    for t in sorted(loadings_by_t.keys()):
        if (t not in mom_short.index) or (t not in mom_long.index):
            continue

        short_t = mom_short.loc[t]
        long_t = mom_long.loc[t]
        if short_t.isna().all() and long_t.isna().all():
            continue

        sig_short, sig_long = build_factor_signal_components(
            mom_short_t=short_t,
            mom_long_t=long_t,
            top_m=top_m,
            select_by_abs=True,
        )

        if np.isclose(sig_short.abs().sum() + sig_long.abs().sum(), 0.0):
            continue

        exposure_t = loadings_by_t[t]
        elig_t = eligibility_mask_by_t[t].reindex(all_tickers).fillna(False).astype(bool)

        rhat_short_t = map_signal_to_etf_prediction(exposure_t, sig_short, elig_t)
        rhat_long_t = map_signal_to_etf_prediction(exposure_t, sig_long, elig_t)

        z_t = compute_volume_zscore_etf(vol_p, t, volume_z_window_periods, all_tickers)
        pi_t = compute_pi_from_volume_z(z_t, a=a, c=c)
        rhat_mix_t, kappa_t = combine_short_long_prediction(rhat_short_t, rhat_long_t, pi_t)

        w_t = _allocate_proportional_long_only(rhat_mix_t, elig_t)
        if np.isclose(w_t.sum(), 0.0):
            continue

        weights_dict[t] = w_t
        diag['rhat_short'][t] = rhat_short_t
        diag['rhat_long'][t] = rhat_long_t
        diag['rhat_mix'][t] = rhat_mix_t
        diag['z'][t] = z_t
        diag['pi'][t] = pi_t
        diag['kappa'][t] = kappa_t
        diag['elig'][t] = elig_t

    weights = pd.DataFrame.from_dict(weights_dict, orient='index').reindex(columns=all_tickers).fillna(0.0).sort_index()
    gross, net, turn, equity, dd = run_backtest(weights, ret_p, cost_bps)

    return {
        'weights': weights,
        'gross': gross,
        'net': net,
        'turnover': turn,
        'equity': equity,
        'drawdown': dd,
        'diag': diag,
    }


def run_interaction_diagnostic(diag, ret_p):
    rows = []

    for t, rshort in diag['rhat_short'].items():
        apply_t = next_period_index(ret_p.index, t)
        if apply_t is None:
            continue

        z = diag['z'][t]
        elig = diag['elig'][t]
        y = ret_p.loc[apply_t].reindex(rshort.index)

        df = pd.DataFrame({
            'x1_short': rshort,
            'z': z,
            'y_next': y,
            'elig': elig,
        })
        df = df[df['elig']].drop(columns=['elig']).dropna()
        if len(df) < 8:
            continue

        x1 = df['x1_short'].values.astype(float)
        z1 = df['z'].values.astype(float)
        y1 = df['y_next'].values.astype(float)
        xint = x1 * z1

        X = np.column_stack([np.ones(len(df)), x1, z1, xint])
        coef = np.linalg.lstsq(X, y1, rcond=None)[0]

        rows.append({
            't': t,
            'n_obs': len(df),
            'beta0': coef[0],
            'beta1_short': coef[1],
            'beta2_z': coef[2],
            'beta3_interaction': coef[3],
        })

    ts = pd.DataFrame(rows).set_index('t').sort_index() if rows else pd.DataFrame()

    if ts.empty:
        return pd.DataFrame(), ts

    out = []
    for col in ['beta0', 'beta1_short', 'beta2_z', 'beta3_interaction']:
        s = ts[col].dropna()
        n = len(s)
        mean = float(s.mean())
        std = float(s.std(ddof=1)) if n > 1 else np.nan
        tstat = mean / (std / np.sqrt(n)) if (n > 1 and std > 0) else np.nan
        out.append({'coef': col, 'mean': mean, 'std': std, 't_stat': tstat, 'n': n})

    summary = pd.DataFrame(out).set_index('coef')
    return summary, ts


In [ ]:
all_tickers = ret_p.columns.tolist()

# EQW baseline（PCA時点の投資可能銘柄に等配分）
eqw_rows = {}
for t, elig in sorted(eligibility_mask_by_t.items()):
    elig = elig.reindex(all_tickers).fillna(False).astype(bool)
    w = pd.Series(0.0, index=all_tickers, dtype=float)
    n = int(elig.sum())
    if n > 0:
        w.loc[elig.index[elig.values]] = 1.0 / n
    eqw_rows[t] = w

weights_eqw = pd.DataFrame.from_dict(eqw_rows, orient='index').reindex(columns=all_tickers).fillna(0.0).sort_index()
gross_eqw, net_eqw, turn_eqw, eqw_equity, eqw_dd = run_backtest(weights_eqw, ret_p, CONFIG['cost_bps'])

# VOLMIX
volmix_res = run_volume_conditioned_strategy(
    ret_p=ret_p,
    vol_p=vol_p,
    loadings_by_t=loadings_by_t,
    eligibility_mask_by_t=eligibility_mask_by_t,
    mom_short=mom_short,
    mom_long=mom_long,
    top_m=CONFIG['top_m_factors'],
    volume_z_window_periods=RUNTIME_VM['volume_z_window_periods'],
    a=CONFIG['volmix_a'],
    c=CONFIG['volmix_c'],
    cost_bps=CONFIG['cost_bps'],
)

net_volmix = volmix_res['net']
turn_volmix = volmix_res['turnover']
volmix_equity = volmix_res['equity']
volmix_dd = volmix_res['drawdown']

print('EQW periods:', len(net_eqw), '| VOLMIX periods:', len(net_volmix))
print('VOLMIX decision periods:', len(volmix_res['weights']))

interaction_summary, interaction_ts = run_interaction_diagnostic(volmix_res['diag'], ret_p)
print('interaction windows:', len(interaction_ts))
display(interaction_summary)


In [ ]:
def calc_metrics(net_ret, turnover, periods_per_year):
    r = net_ret.dropna()
    if len(r) == 0:
        return pd.Series({'CAGR': np.nan, 'Vol': np.nan, 'Sharpe': np.nan, 'MDD': np.nan, 'Turnover': np.nan, 'Hit': np.nan})

    years = len(r) / float(periods_per_year)
    cagr = (1.0 + r).prod() ** (1.0 / years) - 1.0 if years > 0 else np.nan
    vol = r.std(ddof=0) * np.sqrt(periods_per_year)
    sharpe = (r.mean() / r.std(ddof=0) * np.sqrt(periods_per_year)) if r.std(ddof=0) > 0 else np.nan

    equity = (1.0 + r).cumprod()
    mdd = (equity / equity.cummax() - 1.0).min()
    to = turnover.reindex(r.index).mean()
    hit = (r > 0).mean()

    return pd.Series({'CAGR': cagr, 'Vol': vol, 'Sharpe': sharpe, 'MDD': mdd, 'Turnover': to, 'Hit': hit})


common_idx = net_eqw.index.intersection(net_volmix.index)
if len(common_idx) == 0:
    raise ValueError('No common period between EQW and VOLMIX.')

net_eqw_c = net_eqw.reindex(common_idx)
net_volmix_c = net_volmix.reindex(common_idx)
turn_eqw_c = turn_eqw.reindex(common_idx)
turn_volmix_c = turn_volmix.reindex(common_idx)

eqw_equity_c = (1.0 + net_eqw_c).cumprod()
volmix_equity_c = (1.0 + net_volmix_c).cumprod()
eqw_dd_c = eqw_equity_c / eqw_equity_c.cummax() - 1.0
volmix_dd_c = volmix_equity_c / volmix_equity_c.cummax() - 1.0

metrics_table = pd.DataFrame({
    'EQW': calc_metrics(net_eqw_c, turn_eqw_c, RUNTIME['periods_per_year']),
    'VOLMIX': calc_metrics(net_volmix_c, turn_volmix_c, RUNTIME['periods_per_year']),
}).T

delta_vs_eqw = pd.DataFrame({
    'Sharpe_delta': [metrics_table.loc['VOLMIX', 'Sharpe'] - metrics_table.loc['EQW', 'Sharpe']],
    'CAGR_delta': [metrics_table.loc['VOLMIX', 'CAGR'] - metrics_table.loc['EQW', 'CAGR']],
    'MDD_delta': [metrics_table.loc['VOLMIX', 'MDD'] - metrics_table.loc['EQW', 'MDD']],
    'Turnover_delta': [metrics_table.loc['VOLMIX', 'Turnover'] - metrics_table.loc['EQW', 'Turnover']],
}, index=['VOLMIX - EQW'])

print(f'Common window: {common_idx.min().date()} -> {common_idx.max().date()} ({len(common_idx)} periods)')
display(metrics_table)
display(delta_vs_eqw)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(eqw_equity_c.index, eqw_equity_c.values, label='EQW', linewidth=2)
axes[0].plot(volmix_equity_c.index, volmix_equity_c.values, label='VOLMIX', linewidth=2)
axes[0].set_title('EQW vs VOLMIX | Equity')
axes[0].legend()

axes[1].plot(eqw_dd_c.index, eqw_dd_c.values, label='EQW', linewidth=1.8)
axes[1].plot(volmix_dd_c.index, volmix_dd_c.values, label='VOLMIX', linewidth=1.8)
axes[1].set_title('EQW vs VOLMIX | Drawdown')
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
# --- Validation policy checks ---
diag = volmix_res['diag']

# 1) pi in [0,1], kappa in [-1,1]
pi_vals = []
kappa_vals = []
for t in diag['pi']:
    pi_t = diag['pi'][t].values.astype(float)
    k_t = diag['kappa'][t].values.astype(float)
    pi_vals.append(pi_t[np.isfinite(pi_t)])
    kappa_vals.append(k_t[np.isfinite(k_t)])

if len(pi_vals) > 0:
    pi_all = np.concatenate(pi_vals)
    assert (pi_all >= -1e-12).all() and (pi_all <= 1.0 + 1e-12).all(), 'pi out of range'

if len(kappa_vals) > 0:
    kappa_all = np.concatenate(kappa_vals)
    assert (kappa_all >= -1.0 - 1e-12).all() and (kappa_all <= 1.0 + 1e-12).all(), 'kappa out of range'

# 2) monotonicity: z increase -> pi nondecreasing, kappa nonincreasing
for t in diag['z']:
    z_t = diag['z'][t].fillna(0.0)
    pi_t = diag['pi'][t].reindex(z_t.index).fillna(0.5)
    k_t = diag['kappa'][t].reindex(z_t.index).fillna(0.0)

    order = np.argsort(z_t.values)
    pi_s = pi_t.values[order]
    k_s = k_t.values[order]

    assert (np.diff(pi_s) >= -1e-10).all(), 'pi monotonicity failed'
    assert (np.diff(k_s) <= 1e-10).all(), 'kappa monotonicity failed'

# 3) time alignment: decision t -> apply t+1
dec = volmix_res['net'].attrs.get('decision_periods', pd.DatetimeIndex([]))
for t in dec:
    apply_t = next_period_index(ret_p.index, t)
    assert apply_t is not None and apply_t > t, 'time alignment failed'

# 4) long-only weight constraints
w = volmix_res['weights']
if not w.empty:
    assert (w.values >= -1e-12).all(), 'negative weight found'
    row_sum = w.sum(axis=1)
    active = row_sum > 1e-12
    if active.any():
        assert np.allclose(row_sum[active].values, 1.0, atol=1e-8), 'active sum != 1'
    if (~active).any():
        assert np.allclose(row_sum[~active].values, 0.0, atol=1e-8), 'inactive sum != 0'

# 5) common period non-empty
assert len(common_idx) > 0, 'common period is empty'

# 6) metrics finite
assert np.isfinite(metrics_table.values).all(), 'metrics contains NaN/inf'

print('All validation checks passed.')


In [ ]:
# Additional diagnostics: pi/kappa time-series and volume-bucket behavior
pi_mean = pd.Series({t: float(diag['pi'][t].mean()) for t in diag['pi']}).sort_index()
kappa_mean = pd.Series({t: float(diag['kappa'][t].mean()) for t in diag['kappa']}).sort_index()

fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
pi_mean.plot(ax=ax[0], linewidth=1.8, color='tab:blue', title='Average $\\pi_t$ (Volume-conditioned reversal probability)')
kappa_mean.plot(ax=ax[1], linewidth=1.8, color='tab:orange', title='Average $\\kappa_t$ (Short-momentum slope)')
plt.tight_layout()
plt.show()

bucket_rows = []
for t in diag['z']:
    z_t = diag['z'][t]
    k_t = diag['kappa'][t]
    rshort_t = diag['rhat_short'][t]
    elig_t = diag['elig'][t]

    df = pd.DataFrame({'z': z_t, 'kappa': k_t, 'rshort': rshort_t, 'elig': elig_t})
    df = df[df['elig']].drop(columns=['elig']).dropna()
    if len(df) < 5:
        continue

    df['bucket'] = pd.qcut(df['z'].rank(method='first'), q=5, labels=False, duplicates='drop') + 1
    g = df.groupby('bucket').agg(kappa_mean=('kappa', 'mean'), abs_short=('rshort', lambda x: float(np.mean(np.abs(x)))))
    g['t'] = t
    bucket_rows.append(g.reset_index())

if bucket_rows:
    bucket_df = pd.concat(bucket_rows, ignore_index=True)
    bucket_summary = bucket_df.groupby('bucket')[['kappa_mean', 'abs_short']].mean()
    display(bucket_summary)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(bucket_summary.index, bucket_summary['kappa_mean'], marker='o', linewidth=1.8)
    ax.set_title('Volume bucket vs average $\\kappa$ (higher bucket = higher volume z)')
    ax.set_xlabel('Volume z bucket (1 low -> 5 high)')
    ax.set_ylabel('Average kappa')
    plt.tight_layout()
    plt.show()


## 8) まとめ

- 本ノートは `EQW` を唯一の基準として、Volume条件付き短期成分反転（VOLMIX）を評価した。
- 固定設定（`a=1.2, c=0.5`）のみであり、最適化探索は行っていない。
- 次の拡張候補:
  1. `a,c` のwalk-forward選定
  2. 1M成分以外（3M等）への適用比較
  3. 別の流動性指標（dollar volume, turnover ratio）での頑健性確認
